# Exposure -> Conversion Join and Attribution Window Comparison

## Beginner-friendly summary
This notebook closes the measurement gap by using a real event-level dataset to build explicit exposure->conversion joins and compare attribution logic across different windows.

### What this notebook does
- Uses a real-world attribution dataset (Criteo Attribution Modeling Dataset)
- Builds an exposure->conversion joined table with clear lookback rules
- Compares attribution methods (last-touch, linear multi-touch, time-decay)
- Compares attribution outputs across multiple windows (1-day, 7-day, 14-day)
- Produces an attribution-ready measurement table for downstream reporting/modeling

### Major steps (in order)
1. Download/load real dataset
2. Prepare exposure and conversion event tables
3. Build windowed exposure->conversion joins
4. Attribution method comparison
5. Attribution window comparison
6. Final measurement-ready output

### Side notes for beginners
- Join window defines how far back an exposure can receive conversion credit.
- Attribution method choice can change channel credit materially.
- This notebook focuses on measurement logic, not churn/CTR model training.


## 1) Setup

> Side note: We keep dependencies lightweight (`pandas`, `numpy`) so the notebook is easy to run.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

## 2) Download Real-World Dataset (Criteo Attribution)

Dataset source:
- [Criteo Attribution Modeling Dataset](https://huggingface.co/datasets/criteo/criteo-attribution-dataset)

Direct file used in this notebook:
- `criteo_attribution_dataset.tsv.gz`

> Side note: The raw file is large. We read a configurable row sample for fast iteration.

In [ ]:
import urllib.request

data_dir = Path('data')
data_dir.mkdir(parents=True, exist_ok=True)

dataset_path = data_dir / 'criteo_attribution_dataset.tsv.gz'
dataset_url = 'https://huggingface.co/datasets/criteo/criteo-attribution-dataset/resolve/main/criteo_attribution_dataset.tsv.gz'

if not dataset_path.exists():
    print('Downloading real-world dataset (this can take time)...')
    urllib.request.urlretrieve(dataset_url, dataset_path)
    print('Download complete:', dataset_path)
else:
    print('Dataset already present:', dataset_path)

## 3) Load and Prepare Event Tables

Expected raw columns (tab-separated, no header):
- `timestamp`
- `uid`
- `campaign`
- `conversion`
- `conversion_timestamp`

> Side note: A single user may have many touchpoints and multiple conversions.

In [ ]:
TARGET_CONVERTED_USERS = 2000
CHUNK_SIZE = 500_000
cols = ['timestamp', 'uid', 'campaign', 'conversion', 'conversion_timestamp', 'conversion_id']

# Pass 1: collect users who have at least one conversion event
converted_users = set()
for chunk in pd.read_csv(dataset_path, sep='\t', compression='gzip', usecols=cols, chunksize=CHUNK_SIZE, low_memory=False):
    chunk['conversion'] = pd.to_numeric(chunk['conversion'], errors='coerce').fillna(0).astype(int)
    conv = chunk[chunk['conversion'] == 1]
    if not conv.empty:
        converted_users.update(conv['uid'].dropna().astype(str).tolist())
    if len(converted_users) >= TARGET_CONVERTED_USERS:
        break

selected_users = set(list(converted_users)[:TARGET_CONVERTED_USERS])
print('Selected converted users:', len(selected_users))

# Pass 2: collect all touchpoints for selected users
parts = []
for chunk in pd.read_csv(dataset_path, sep='\t', compression='gzip', usecols=cols, chunksize=CHUNK_SIZE, low_memory=False):
    chunk['uid'] = chunk['uid'].astype(str)
    keep = chunk[chunk['uid'].isin(selected_users)]
    if not keep.empty:
        parts.append(keep)

if not parts:
    raise RuntimeError('No rows found for selected converted users. Try increasing TARGET_CONVERTED_USERS.')

raw = pd.concat(parts, ignore_index=True)

raw['timestamp'] = pd.to_numeric(raw['timestamp'], errors='coerce')
raw['conversion_timestamp'] = pd.to_numeric(raw['conversion_timestamp'], errors='coerce')
raw['conversion'] = pd.to_numeric(raw['conversion'], errors='coerce').fillna(0).astype(int)
raw['conversion_id'] = pd.to_numeric(raw['conversion_id'], errors='coerce')
raw = raw.dropna(subset=['timestamp', 'uid', 'campaign']).copy()
raw['ts_exposure'] = pd.to_datetime(raw['timestamp'], unit='s', errors='coerce')

# Exposure table
exposures = raw[['uid', 'campaign', 'timestamp', 'ts_exposure']].rename(columns={'uid': 'user_id', 'campaign': 'channel'})
exposures['exposure_id'] = np.arange(len(exposures), dtype=np.int64)

# Conversion table: one row per distinct conversion event
conv_rows = raw[(raw['conversion'] == 1) & (raw['conversion_timestamp'].notna())].copy()
conv_rows = conv_rows[conv_rows['conversion_timestamp'] >= conv_rows['timestamp']]

conversions = conv_rows[['uid', 'conversion_id', 'conversion_timestamp']].rename(columns={'uid': 'user_id'}).drop_duplicates().reset_index(drop=True)
conversions['ts_conversion'] = pd.to_datetime(conversions['conversion_timestamp'], unit='s', errors='coerce')
conversions = conversions.dropna(subset=['ts_conversion']).reset_index(drop=True)
conversions['conversion_id'] = np.arange(len(conversions), dtype=np.int64)
conversions['conversion_value'] = 1.0

print('Filtered raw rows:', len(raw))
print('Exposure rows:', len(exposures))
print('Conversion rows:', len(conversions))
print('Unique users in filtered set:', exposures['user_id'].nunique())

display(exposures.head(5))
display(conversions.head(5))


## 4) Exposure -> Conversion Join (Windowed)

Join rule:
- same `user_id`
- exposure occurred before conversion
- exposure within lookback window (in days)

> Side note: This is the auditable core of measurement pipelines.

In [ ]:
def build_join_table(exposures_df: pd.DataFrame, conversions_df: pd.DataFrame, lookback_days: int) -> pd.DataFrame:
    merged = conversions_df.merge(exposures_df, on='user_id', how='left')
    delta_hours = (merged['ts_conversion'] - merged['ts_exposure']).dt.total_seconds() / 3600

    valid = merged[(delta_hours >= 0) & (delta_hours <= lookback_days * 24)].copy()
    if valid.empty:
        return pd.DataFrame(columns=[
            'lookback_days', 'conversion_id', 'user_id', 'ts_conversion', 'conversion_value',
            'exposure_id', 'channel', 'ts_exposure', 'hours_before_conversion'
        ])

    valid['hours_before_conversion'] = delta_hours.loc[valid.index]
    valid['lookback_days'] = lookback_days

    cols = [
        'lookback_days', 'conversion_id', 'user_id', 'ts_conversion', 'conversion_value',
        'exposure_id', 'channel', 'ts_exposure', 'hours_before_conversion'
    ]
    return valid[cols].sort_values(['conversion_id', 'ts_exposure']).reset_index(drop=True)

joined_7d = build_join_table(exposures, conversions, lookback_days=7)
print('Joined rows (7-day):', len(joined_7d))
print('Unique conversions covered (7-day):', joined_7d['conversion_id'].nunique())
print('Coverage % (7-day):', round(100 * joined_7d['conversion_id'].nunique() / max(len(conversions), 1), 2))
display(joined_7d.head(10))


## 4b) Coverage Summary (Explicit)

> Side note: Coverage tells us what fraction of conversions can be linked to at least one prior exposure in the selected lookback window.


In [ ]:
total_conversions = int(conversions['conversion_id'].nunique())
matched_conversions_7d = int(joined_7d['conversion_id'].nunique())
coverage_pct_7d = round(100 * matched_conversions_7d / max(total_conversions, 1), 2)

coverage_summary = pd.DataFrame([{
    'lookback_days': 7,
    'total_conversions': total_conversions,
    'matched_conversions': matched_conversions_7d,
    'coverage_pct': coverage_pct_7d
}])

print(f"Coverage formula: matched_conversions / total_conversions = {matched_conversions_7d} / {total_conversions} = {coverage_pct_7d}%")
display(coverage_summary)


## 5) Attribution Method Comparison

Methods:
- Last-touch
- Linear multi-touch
- Time-decay

> Side note: Method choice changes channel credit. This is why teams compare methods, not just one.

In [ ]:
def attrib_last_touch(join_df: pd.DataFrame) -> pd.DataFrame:
    idx = join_df.groupby('conversion_id')['ts_exposure'].idxmax()
    out = join_df.loc[idx, ['channel', 'conversion_value']].copy()
    out['credit'] = out['conversion_value']
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_linear(join_df: pd.DataFrame) -> pd.DataFrame:
    cnt = join_df.groupby('conversion_id')['exposure_id'].transform('count')
    out = join_df[['channel', 'conversion_value']].copy()
    out['credit'] = out['conversion_value'] / cnt
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_time_decay(join_df: pd.DataFrame, half_life_hours: float = 48.0) -> pd.DataFrame:
    out = join_df[['conversion_id', 'channel', 'conversion_value', 'hours_before_conversion']].copy()
    out['w'] = np.exp(-np.log(2) * out['hours_before_conversion'] / half_life_hours)
    norm = out.groupby('conversion_id')['w'].transform('sum')
    out['credit'] = out['conversion_value'] * out['w'] / norm
    return out.groupby('channel', as_index=False)['credit'].sum()

last_touch_7d = attrib_last_touch(joined_7d).rename(columns={'credit': 'last_touch_credit'})
linear_7d = attrib_linear(joined_7d).rename(columns={'credit': 'linear_credit'})
decay_7d = attrib_time_decay(joined_7d).rename(columns={'credit': 'time_decay_credit'})

comparison_7d = last_touch_7d.merge(linear_7d, on='channel', how='outer').merge(decay_7d, on='channel', how='outer').fillna(0)
comparison_7d = comparison_7d.sort_values('last_touch_credit', ascending=False)
display(comparison_7d)

## 6) Attribution Window Comparison (1-day vs 7-day vs 14-day)

> Side note: Wider windows include more exposures, which often redistributes channel credit and changes decisions.

In [ ]:
def summarize_window(window_days: int) -> pd.DataFrame:
    j = build_join_table(exposures, conversions, lookback_days=window_days)
    if j.empty:
        return pd.DataFrame(columns=['window_days', 'method', 'channel', 'credit'])

    lt = attrib_last_touch(j).assign(method='last_touch', window_days=window_days)
    li = attrib_linear(j).assign(method='linear', window_days=window_days)
    td = attrib_time_decay(j).assign(method='time_decay', window_days=window_days)
    out = pd.concat([lt, li, td], ignore_index=True)
    return out[['window_days', 'method', 'channel', 'credit']]

window_results = pd.concat([summarize_window(w) for w in [1, 7, 14]], ignore_index=True)

credit_pivot = window_results.pivot_table(
    index='channel',
    columns=['window_days', 'method'],
    values='credit',
    aggfunc='sum'
).fillna(0)

totals = window_results.groupby(['window_days', 'method'], as_index=False)['credit'].sum()

display(credit_pivot)
display(totals)

## 7) Measurement-Ready Output Table

This table is what downstream dashboards/analytics/modeling layers can consume.

> Side note: In production this is often saved to a warehouse/Delta table with versioned logic.

In [ ]:
measurement_table = build_join_table(exposures, conversions, lookback_days=7)
print('Rows:', len(measurement_table))
print('Unique conversions:', measurement_table['conversion_id'].nunique())
display(measurement_table.head(15))

## 8) Key Learning

- This notebook uses real-world event data and explicit joins to build an attribution-ready measurement layer.
- Both attribution method and attribution window have material impact on channel credit.
- This is a practical bridge between raw event logs and decision-ready marketing measurement.